In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.1"

import numpy as np
import matplotlib.pyplot as plt
import cluster as cl
import jax
import jzfmm
import aegis
import json
from dataclasses import replace

%load_ext autoreload
import cluster_plots as cp

In [ ]:
prof = aegis.profiles.NFWProfile(6., m200c=1e14)
print(prof.rs, prof.r200c)
print(prof.tcirc(prof.r200c, inyears=True) / 1e9)
print(prof.tcirc(prof.r200c) / prof.tcirc(prof.rs))

In [ ]:
print(prof.vcirc(500))

In [ ]:
def closest(x1, x2):
    r2mat = np.sum((x1[...,:,None,:] - x2[...,None,:,:])**2, axis=-1)
    return np.sqrt(np.min(r2mat, axis=-1))
def l10mtot(l10m):
    return np.log10(np.sum(10.**l10m, axis=-1))

def plot_history(file):
    res = np.load(file)
    history = res["history"]
    parameters = history["parameters"]
    target = res["target_parameters"]
    log10_mass = np.log10(np.asarray(cl.mass_from_exp.jit(parameters["exp_m"])))
    target_log10_mass = np.log10(np.asarray(cl.mass_from_exp.jit(target["exp_m"])))
    # scale_radius = np.asarray(cl.radius_from_exp.jit(parameters["exp_rs"]))
    # target_scale_radius = np.asarray(cl.radius_from_exp.jit(target["exp_rs"]))
    concentration = np.asarray(cl.concentration_from_log10.jit(
        parameters["log10_concentration"]
    ))
    target_concentration = np.asarray(cl.concentration_from_log10.jit(
        target["log10_concentration"]
    ))

    fig, axs = plt.subplots(5,1, figsize=(6,10), sharex=True)
    fig.subplots_adjust(hspace=0.02)
    axs[0].semilogy(history["step"], history["loss"])
    axs[0].axhline(res["optimal_loss"], ls="dotted", color="black")

    axs[1].plot(history["step"], log10_mass)
    axs[1].plot(history["step"], l10mtot(log10_mass), color="black")
    for v in target_log10_mass:
        axs[1].axhline(v, color="grey", ls="dotted")
    axs[1].axhline(l10mtot(target_log10_mass), color="black", ls="dotted")

    # axs[2].plot(history["step"], np.log10(scale_radius*1e3))
    # for v in target_scale_radius:
    #     axs[2].axhline(np.log10(v*1e3), color="grey", ls="dotted")
    axs[2].plot(history["step"], concentration)
    for v in target_concentration:
        axs[2].axhline(v, color="grey", ls="dotted")
    
    rclosest = closest(parameters["position"], target["position"])
    axs[3].semilogy(history["step"], rclosest*1e3)

    vclosest = closest(parameters["velocity"], target["velocity"])
    axs[4].semilogy(history["step"], vclosest*1e3)


    axs[0].set_ylabel("loss")
    axs[1].set_ylabel(r"$\log_{10} (M/M_{\odot})$")
    axs[2].set_ylabel(r"$\log_{10} (r_s/\mathrm{kpc})$")
    axs[3].set_ylabel(r"$r_{\mathrm{closest}}$ [kpc]")
    axs[4].set_ylabel(r"$v_{\mathrm{closest}}$ [km/s]")

    axs[1].set_ylim(11.5,13.5)

In [ ]:
plot_history("logs/sim_109.npz")

In [ ]:
sim = 108
noutputs = 6

res = np.load(f"logs/sim_{sim}.npz")
cfg = cl.Config.from_json(f"logs/sim_{sim}.json")

isel = np.argmin(res["history"]["loss"])
it_target = iter(cl.get_particles(res["target_parameters"], cfg, seed=cfg.target_particle_seed, outputs=noutputs))
it_sim = iter(cl.get_particles(res["history"]["parameters"][isel], cfg, seed=cfg.model_particle_seed, outputs=noutputs))

fig, axs = plt.subplots(1,2, figsize=(10,4.5))
for i in range(noutputs):
    tgyr,particles_target = next(it_target)
    tgyr,particles_sim = next(it_sim)

    axs[0].scatter(particles_target.pos[:,0], particles_target.pos[:,1], marker=".", alpha=0.01)
    axs[1].scatter(particles_sim.pos[:,0], particles_sim.pos[:,1], marker=".", alpha=0.01)


axs[0].set_title("target")
axs[1].set_title("infered")

for ax in axs:
    ax.set_xlim(-2000,2000)
    ax.set_ylim(-2000,2000)

plt.figure()
plt.scatter(particles_target.pos[:,0], particles_target.pos[:,1], marker=".", alpha=0.03, color="blue")
plt.scatter(particles_sim.pos[:,0], particles_sim.pos[:,1], marker=".", alpha=0.03, color="red")

In [ ]:
%autoreload

In [ ]:
fig, axs = cp.plot_runs(
    [227, 159, 93], #223
    noutputs=5,
    nstates=8,
)
plt.savefig("logs/successful_reconstructions.pdf", bbox_inches="tight")
plt.savefig("logs/successful_reconstructions.png", bbox_inches="tight")

In [ ]:
fig, axs = cp.plot_runs(
    [100, 194, "convergence_grid/run_4395"],
    noutputs=5,
    nstates=8,
)
plt.savefig("logs/failed_reconstructions.pdf", bbox_inches="tight")
plt.savefig("logs/failed_reconstructions.png", bbox_inches="tight")

In [ ]:
summary = np.load(
    "logs/convergence_grid/analysis/summary.npz"
)["summary"]

In [ ]:
%autoreload

fig, axs = cp.plot_convergence_summary(summary)
plt.savefig("logs/convergence_grid.pdf", bbox_inches="tight")